# 样本向量化（查询城市 & 目标城市）

- 输入：`samples_w192_s6/query` 与 `samples_w192_s6/target` 下的 `.npz`
- 模型：Chronos Pipeline（时序 embedding）
- 输出：`vector_db_w192_s6/query`、`vector_db_w192_s6/target`
  - `embeddings.npy`, `id_mapping.pkl`, `meta.json`, `faiss_index.bin`, `metadata.json`

## 1. 参数与路径
- `SAMPLES_ROOT`：样本根目录（前一步生成）
- `VECTOR_ROOT`：向量库输出根目录
- pooling 可选 `mean`/`last`/`max`/`weighted_mean`
- `chronos_model_path` 指向本地 Chronos 模型目录

下载地址：https://modelscope.cn/models/better464/times

In [1]:
from pathlib import Path

WINDOW = 192
INPUT_LEN = 168
STEP = 6

# 目录配置（按需修改）
SAMPLES_ROOT = Path('./shaanxi_cities_temperature/samples_w192_s6')
VECTOR_ROOT = Path('./shaanxi_cities_temperature/vector_db_w192_s6')
VECTOR_ROOT.mkdir(parents=True, exist_ok=True)

# 模型与参数
CHRONOS_MODEL_PATH = 'chronos'
POOLING = 'last'
BATCH_SIZE = 256

print('SAMPLES_ROOT:', SAMPLES_ROOT)
print('VECTOR_ROOT :', VECTOR_ROOT)

SAMPLES_ROOT: shaanxi_cities_temperature/samples_w192_s6
VECTOR_ROOT : shaanxi_cities_temperature/vector_db_w192_s6


## 2. 向量化工具

chronos库安装：pip install git+https://github.com/amazon-science/chronos-forecasting.git

In [2]:
import json
import pickle
from datetime import datetime
from typing import Dict, List, Optional

import faiss
import numpy as np
import torch
from chronos import ChronosPipeline
from tqdm import tqdm
#chronos库安装：pip install git+https://github.com/amazon-science/chronos-forecasting.git
CITY_SLUG = {
    "西安": "xian", "郑州": "zhengzhou", "太原": "taiyuan", "石家庄": "shijiazhuang",
    "济南": "jinan", "呼和浩特": "huhehaote", "银川": "yinchuan", "兰州": "lanzhou",
    "西宁": "xining", "大同": "datong", "汉中": "hanzhong"
}

def _safe_slug(name: str) -> str:
    if name in CITY_SLUG:
        return CITY_SLUG[name]
    cleaned = ''.join(ch if ch.isascii() and ch.isalnum() else '_' for ch in name).strip('_')
    return cleaned or 'city'

class CityTempVectorDatabase:
    def __init__(self, samples_npz_dir: Path, vector_db_dir: Path, pooling='last', batch_size=256, chronos_model_path=CHRONOS_MODEL_PATH, device=None):
        self.samples_npz_dir = Path(samples_npz_dir)
        self.vector_db_dir = Path(vector_db_dir)
        self.pooling = pooling
        self.batch_size = batch_size
        self.device = device or ('cuda:0' if torch.cuda.is_available() else 'cpu')
        self.chronos_model_path = chronos_model_path
        self.vector_db_dir.mkdir(parents=True, exist_ok=True)

        print(f"Device: {self.device} | Pooling: {self.pooling}")
        self._init_chronos()

        self.embeddings = None
        self.sample_id_to_idx = {}
        self.idx_to_sample_id = {}
        self.meta: List[Dict] = []
        self.faiss_index = None

    def _init_chronos(self):
        print('Loading Chronos...')
        self.chronos = ChronosPipeline.from_pretrained(
            self.chronos_model_path,
            device_map=self.device,
            dtype=torch.float32,
        )
        print('Chronos loaded')

    def _pool(self, emb: torch.Tensor):
        if self.pooling == 'mean':
            return emb.mean(dim=1)
        if self.pooling == 'last':
            return emb[:, -1, :]
        if self.pooling == 'max':
            return emb.max(dim=1)[0]
        if self.pooling == 'weighted_mean':
            seq_len = emb.shape[1]
            w = torch.arange(1, seq_len + 1, device=emb.device, dtype=torch.float32)
            w = w / w.sum()
            return (emb * w.view(1, -1, 1)).sum(dim=1)
        raise ValueError(f'Unknown pooling {self.pooling}')

    def _iter_npz(self):
        return sorted(self.samples_npz_dir.glob('*_samples_w192_s6.npz'))

    def extract_embeddings(self):
        all_emb, sample_ids, meta = [], [], []
        files = self._iter_npz()
        if not files:
            raise FileNotFoundError(f'No npz in {self.samples_npz_dir}')
        for npz_path in files:
            city_name = npz_path.stem.replace('_samples_w192_s6', '')
            slug = _safe_slug(city_name)
            with np.load(npz_path) as data:
                inputs = data['input']
                start_times = data['start_time']
                n = inputs.shape[0]
                for s in tqdm(range(0, n, self.batch_size), desc=city_name):
                    e = min(s + self.batch_size, n)
                    batch = torch.tensor(inputs[s:e], dtype=torch.float32)
                    with torch.no_grad():
                        emb_raw, _ = self.chronos.embed(batch)
                        pooled = self._pool(emb_raw).cpu().numpy().astype(np.float32)
                    for i, emb in enumerate(pooled):
                        ts_raw = start_times[s + i]
                        ts_hour = np.datetime64(ts_raw, 'h')
                        ts_str = np.datetime_as_string(ts_hour, unit='h').replace('-', '').replace('T', '')
                        sid = f"{slug}_{ts_str}"
                        idx = len(sample_ids)
                        sample_ids.append(sid)
                        all_emb.append(emb)
                        meta.append({
                            'sample_id': sid,
                            'city': city_name,
                            'idx_in_city': s + i,
                            'start_time': str(start_times[s + i]),
                            'npz_path': str(npz_path),
                        })
        self.embeddings = np.stack(all_emb).astype(np.float32)
        self.sample_id_to_idx = {sid: i for i, sid in enumerate(sample_ids)}
        self.idx_to_sample_id = {i: sid for i, sid in enumerate(sample_ids)}
        self.meta = meta
        print('Total samples:', len(self.embeddings), 'dim:', self.embeddings.shape[1])

    def build_faiss_index(self, index_type='IP'):
        if self.embeddings is None:
            raise ValueError('Run extract_embeddings first')
        dim = self.embeddings.shape[1]
        if index_type == 'IP':
            norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
            emb = self.embeddings / (norms + 1e-8)
            index = faiss.IndexFlatIP(dim)
            index.add(emb.astype(np.float32))
            self.embeddings = emb
        elif index_type == 'L2':
            index = faiss.IndexFlatL2(dim)
            index.add(self.embeddings)
        else:
            raise ValueError('index_type only IP or L2')
        self.faiss_index = index
        print(f'FAISS index built: {len(self.embeddings)} vectors, dim={dim}, type={index_type}')

    def save(self):
        if self.embeddings is None:
            raise ValueError('No embeddings to save')
        self.vector_db_dir.mkdir(parents=True, exist_ok=True)
        np.save(self.vector_db_dir / 'embeddings.npy', self.embeddings)
        with open(self.vector_db_dir / 'id_mapping.pkl', 'wb') as f:
            pickle.dump({'sample_id_to_idx': self.sample_id_to_idx, 'idx_to_sample_id': self.idx_to_sample_id}, f)
        with open(self.vector_db_dir / 'meta.json', 'w') as f:
            json.dump(self.meta, f, ensure_ascii=False, indent=2)
        if self.faiss_index is not None:
            faiss.write_index(self.faiss_index, str(self.vector_db_dir / 'faiss_index.bin'))
        metadata = {
            'n_samples': len(self.embeddings),
            'embedding_dim': self.embeddings.shape[1],
            'index_type': type(self.faiss_index).__name__ if self.faiss_index else None,
            'chronos_model_path': self.chronos_model_path,
            'pooling': self.pooling,
            'window': WINDOW,
            'input_len': INPUT_LEN,
            'step': STEP,
            'created_at': datetime.now().isoformat(),
            'device': self.device,
        }
        with open(self.vector_db_dir / 'metadata.json', 'w') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)
        print('Saved to', self.vector_db_dir)

/usr/local/miniconda3/envs/timerag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. 生成查询城市向量库
输入目录：`samples_w192_s6/query`
输出目录：`vector_db_w192_s6/query`

In [3]:
query_db = CityTempVectorDatabase(
    samples_npz_dir=SAMPLES_ROOT / 'query',
    vector_db_dir=VECTOR_ROOT / 'query',
    pooling=POOLING,
    batch_size=BATCH_SIZE,
    chronos_model_path=CHRONOS_MODEL_PATH,
)
query_db.extract_embeddings()
query_db.build_faiss_index('IP')
query_db.save()

Device: cuda:0 | Pooling: last
Loading Chronos...
Chronos loaded


银川: 100%|██████████| 189/189 [15:20<00:00,  4.87s/it]


Total samples: 481810 dim: 1024
FAISS index built: 481810 vectors, dim=1024, type=IP
Saved to shaanxi_cities_temperature/vector_db_w192_s6/query


## 4. 生成目标城市向量库（汉中 2024）
输入目录：`samples_w192_s6/target`
输出目录：`vector_db_w192_s6/target`

In [4]:
target_db = CityTempVectorDatabase(
    samples_npz_dir=SAMPLES_ROOT / 'target',
    vector_db_dir=VECTOR_ROOT / 'target',
    pooling=POOLING,
    batch_size=BATCH_SIZE,
    chronos_model_path=CHRONOS_MODEL_PATH,
)
target_db.extract_embeddings()
target_db.build_faiss_index('IP')
target_db.save()

Device: cuda:0 | Pooling: last
Loading Chronos...
Chronos loaded


汉中: 100%|██████████| 6/6 [00:26<00:00,  4.48s/it]

Total samples: 1433 dim: 1024
FAISS index built: 1433 vectors, dim=1024, type=IP
Saved to shaanxi_cities_temperature/vector_db_w192_s6/target


## 5. 查看输出

In [5]:
import glob
print('向量库文件：')
for f in sorted(glob.glob(str(VECTOR_ROOT / '**/*'), recursive=True)):
    print(f)

向量库文件：
shaanxi_cities_temperature/vector_db_w192_s6/query
shaanxi_cities_temperature/vector_db_w192_s6/query/embeddings.npy
shaanxi_cities_temperature/vector_db_w192_s6/query/faiss_index.bin
shaanxi_cities_temperature/vector_db_w192_s6/query/id_mapping.pkl
shaanxi_cities_temperature/vector_db_w192_s6/query/meta.json
shaanxi_cities_temperature/vector_db_w192_s6/query/metadata.json
shaanxi_cities_temperature/vector_db_w192_s6/target
shaanxi_cities_temperature/vector_db_w192_s6/target/embeddings.npy
shaanxi_cities_temperature/vector_db_w192_s6/target/faiss_index.bin
shaanxi_cities_temperature/vector_db_w192_s6/target/id_mapping.pkl
shaanxi_cities_temperature/vector_db_w192_s6/target/meta.json
shaanxi_cities_temperature/vector_db_w192_s6/target/metadata.json
